# 06. Residual routing — exact mHC and Kimi K3 Attention Residuals

Only tensor width, batch size, and token count are reduced.

Two residual architectures are implemented with their published/released topology:

1. DeepSeek-V4 mHC
   - `hc_mult = 4`
   - separate attention and FFN hyper-connections
   - exact `pre`, `post`, `comb` parameterization
   - 20 Sinkhorn iterations
   - persistent four-stream residual state

2. Kimi K3 Attention Residuals
   - 93 decoder-layer sites
   - block size 12
   - layer-specific attention and MLP residual projections
   - exact block-bank update order
   - final output Attention Residual mix

The branch networks are narrow so the notebook is runnable on CPU, but residual topology is not shortened.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
torch.set_num_threads(min(2, torch.get_num_threads()))
device = torch.device("cpu")
print("device:", device)

## 1. DeepSeek-V4 mHC mapping

In [ ]:
class UnweightedRMSNorm(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, x):
        x_float = x.float()
        inverse_rms = torch.rsqrt(
            x_float.square().mean(dim=-1, keepdim=True)
            + self.eps
        )
        return (x_float * inverse_rms).to(x.dtype)


class DeepSeekV4HyperConnection(nn.Module):
    def __init__(
        self,
        hidden_dim,
        hc_mult=4,
        sinkhorn_iterations=20,
        eps=1e-6,
    ):
        super().__init__()

        self.hc_mult = hc_mult
        self.sinkhorn_iterations = sinkhorn_iterations
        self.eps = eps

        self.input_norm = UnweightedRMSNorm(eps)

        output_size = (2 + hc_mult) * hc_mult

        self.mapping_weight = nn.Parameter(
            torch.empty(
                output_size,
                hc_mult * hidden_dim,
            )
        )
        self.base = nn.Parameter(
            torch.empty(output_size)
        )
        self.scale = nn.Parameter(
            torch.empty(3)
        )

        nn.init.normal_(self.mapping_weight, std=0.02)
        nn.init.zeros_(self.base)
        nn.init.constant_(self.scale, 1e-2)

    def forward(self, hidden_streams):
        hc = self.hc_mult

        flat = hidden_streams.flatten(start_dim=2)
        flat = self.input_norm(flat.float())

        mapping = F.linear(
            flat,
            self.mapping_weight.float(),
        )
        pre_logits, post_logits, comb_logits = mapping.split(
            [hc, hc, hc * hc],
            dim=-1,
        )

        pre_base, post_base, comb_base = self.base.split(
            [hc, hc, hc * hc]
        )
        pre_scale, post_scale, comb_scale = self.scale.unbind(0)

        pre = (
            torch.sigmoid(
                pre_logits * pre_scale + pre_base
            )
            + self.eps
        )
        post = 2.0 * torch.sigmoid(
            post_logits * post_scale + post_base
        )

        comb_logits = (
            comb_logits.view(
                *comb_logits.shape[:-1],
                hc,
                hc,
            )
            * comb_scale
            + comb_base.view(hc, hc)
        )

        comb = (
            torch.softmax(comb_logits, dim=-1)
            + self.eps
        )
        comb = comb / (
            comb.sum(dim=-2, keepdim=True)
            + self.eps
        )

        for _ in range(self.sinkhorn_iterations - 1):
            comb = comb / (
                comb.sum(dim=-1, keepdim=True)
                + self.eps
            )
            comb = comb / (
                comb.sum(dim=-2, keepdim=True)
                + self.eps
            )

        collapsed = (
            pre.unsqueeze(-1)
            * hidden_streams
        ).sum(dim=2)

        return (
            post.to(hidden_streams.dtype),
            comb.to(hidden_streams.dtype),
            collapsed.to(hidden_streams.dtype),
        )


def apply_hyper_connection(
    hidden_streams,
    sublayer,
    hyper_connection,
):
    post, comb, collapsed = hyper_connection(hidden_streams)

    branch_output = sublayer(collapsed)

    mixed_streams = torch.einsum(
        "btij,btjd->btid",
        comb,
        hidden_streams,
    )
    written_branch = (
        post.unsqueeze(-1)
        * branch_output.unsqueeze(2)
    )

    return mixed_streams + written_branch

## 2. Persistent 43-layer V4-Flash residual stack

In [ ]:
class NarrowAttentionBranch(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.norm = nn.RMSNorm(hidden_dim)
        self.projection = nn.Linear(
            hidden_dim,
            hidden_dim,
            bias=False,
        )

    def forward(self, x):
        return self.projection(self.norm(x))


class NarrowFFNBranch(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.norm = nn.RMSNorm(hidden_dim)
        self.gate = nn.Linear(
            hidden_dim,
            2 * hidden_dim,
            bias=False,
        )
        self.value = nn.Linear(
            hidden_dim,
            2 * hidden_dim,
            bias=False,
        )
        self.output = nn.Linear(
            2 * hidden_dim,
            hidden_dim,
            bias=False,
        )

    def forward(self, x):
        normalized = self.norm(x)
        gate = F.silu(self.gate(normalized))
        value = self.value(normalized)
        return self.output(gate * value)


class MHCDecoderLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.attention_hc = DeepSeekV4HyperConnection(
            hidden_dim,
            hc_mult=4,
            sinkhorn_iterations=20,
        )
        self.ffn_hc = DeepSeekV4HyperConnection(
            hidden_dim,
            hc_mult=4,
            sinkhorn_iterations=20,
        )

        self.attention = NarrowAttentionBranch(hidden_dim)
        self.ffn = NarrowFFNBranch(hidden_dim)

    def forward(self, hidden_streams):
        hidden_streams = apply_hyper_connection(
            hidden_streams,
            self.attention,
            self.attention_hc,
        )
        hidden_streams = apply_hyper_connection(
            hidden_streams,
            self.ffn,
            self.ffn_hc,
        )
        return hidden_streams


class MHCFlashResidualStack(nn.Module):
    def __init__(
        self,
        hidden_dim=8,
        depth=43,
        hc_mult=4,
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.hc_mult = hc_mult

        self.input_expansion = nn.Linear(
            hidden_dim,
            hc_mult * hidden_dim,
            bias=False,
        )
        self.layers = nn.ModuleList(
            [
                MHCDecoderLayer(hidden_dim)
                for _ in range(depth)
            ]
        )

        self.head_norm = UnweightedRMSNorm()
        self.head_weight = nn.Parameter(
            torch.empty(
                hc_mult,
                hc_mult * hidden_dim,
            )
        )
        self.head_base = nn.Parameter(
            torch.zeros(hc_mult)
        )
        self.head_scale = nn.Parameter(
            torch.full((1,), 1e-2)
        )
        nn.init.normal_(self.head_weight, std=0.02)

    def collapse_head(self, hidden_streams):
        flat = self.head_norm(
            hidden_streams.flatten(2).float()
        )
        logits = F.linear(
            flat,
            self.head_weight.float(),
        )
        weights = (
            torch.sigmoid(
                logits * self.head_scale.float()
                + self.head_base.float()
            )
            + 1e-6
        )
        return (
            weights.unsqueeze(-1)
            * hidden_streams
        ).sum(dim=2)

    def forward(self, hidden):
        batch_size, sequence_length, hidden_dim = hidden.shape

        hidden_streams = self.input_expansion(hidden)
        hidden_streams = hidden_streams.view(
            batch_size,
            sequence_length,
            self.hc_mult,
            hidden_dim,
        )

        for layer in self.layers:
            hidden_streams = layer(hidden_streams)

        return self.collapse_head(hidden_streams), hidden_streams


mhc_model = MHCFlashResidualStack().to(device)

assert len(mhc_model.layers) == 43
assert mhc_model.hc_mult == 4
assert (
    mhc_model.layers[0]
    .attention_hc
    .sinkhorn_iterations
    == 20
)

mhc_input = torch.randn(1, 2, 8, device=device)
mhc_output, mhc_streams = mhc_model(mhc_input)

test_post, test_comb, _ = (
    mhc_model.layers[0]
    .attention_hc(mhc_streams.detach())
)

row_error = (
    test_comb.sum(dim=-1) - 1
).abs().max()
column_error = (
    test_comb.sum(dim=-2) - 1
).abs().max()

print("V4 residual depth:", len(mhc_model.layers))
print("mHC streams:", mhc_streams.shape)
print("Sinkhorn row error:", row_error.item())
print("Sinkhorn column error:", column_error.item())

## 3. Exact Kimi K3 Attention Residual primitive

In [ ]:
class KimiRMSNorm(nn.Module):
    def __init__(self, hidden_dim, eps=1e-5):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(hidden_dim)
        )
        self.variance_epsilon = eps

    def forward(self, x):
        x_float = x.float()
        variance = x_float.square().mean(
            dim=-1,
            keepdim=True,
        )
        normalized = x_float * torch.rsqrt(
            variance + self.variance_epsilon
        )
        return (
            normalized
            * self.weight.float()
        ).to(x.dtype)


def apply_kimi_attention_residual(
    prefix_sum,
    block_residual,
    score_projection,
    score_norm,
):
    values = torch.cat(
        [
            block_residual,
            prefix_sum.unsqueeze(1),
        ],
        dim=1,
    )

    values_float = values.float()
    variance = values_float.square().mean(
        dim=-1,
        keepdim=True,
    )
    normalized = values_float * torch.rsqrt(
        variance + score_norm.variance_epsilon
    )

    score_weight = (
        score_norm.weight.float()
        * score_projection.weight.squeeze(0).float()
    )
    scores = (
        normalized * score_weight
    ).sum(dim=-1)

    probabilities = scores.softmax(
        dim=-1
    ).unsqueeze(1)

    mixed = torch.matmul(
        probabilities,
        values_float,
    ).squeeze(1)

    return mixed.to(values.dtype)

## 4. Full 93-site K3 Attention Residual schedule

In [ ]:
class K3ResidualSite(nn.Module):
    def __init__(
        self,
        layer_index,
        hidden_dim=8,
        block_size=12,
    ):
        super().__init__()

        self.layer_index = layer_index
        self.block_size = block_size

        self.self_attention_res_norm = KimiRMSNorm(hidden_dim)
        self.mlp_res_norm = KimiRMSNorm(hidden_dim)

        self.self_attention_res_proj = nn.Linear(
            hidden_dim,
            1,
            bias=False,
        )
        self.mlp_res_proj = nn.Linear(
            hidden_dim,
            1,
            bias=False,
        )

        self.input_norm = KimiRMSNorm(hidden_dim)
        self.post_attention_norm = KimiRMSNorm(hidden_dim)

        self.attention_branch = nn.Linear(
            hidden_dim,
            hidden_dim,
            bias=False,
        )
        self.mlp_branch = nn.Sequential(
            nn.Linear(hidden_dim, 2 * hidden_dim, bias=False),
            nn.SiLU(),
            nn.Linear(2 * hidden_dim, hidden_dim, bias=False),
        )

    def forward(
        self,
        hidden_states,
        block_residual,
    ):
        batch_size, sequence_length, hidden_dim = hidden_states.shape

        prefix_sum = hidden_states

        if block_residual.size(1) > 0:
            hidden_states = apply_kimi_attention_residual(
                prefix_sum.reshape(-1, hidden_dim),
                block_residual,
                self.self_attention_res_proj,
                self.self_attention_res_norm,
            )
            hidden_states = hidden_states.view(
                batch_size,
                sequence_length,
                hidden_dim,
            )

        if self.layer_index % self.block_size == 0:
            bank_entry = prefix_sum.reshape(
                -1,
                hidden_dim,
            ).unsqueeze(1)
            block_residual = torch.cat(
                [block_residual, bank_entry],
                dim=1,
            )
            prefix_sum = None

        attention_input = self.input_norm(hidden_states)
        attention_output = self.attention_branch(attention_input)

        if prefix_sum is None:
            prefix_sum = attention_output
        else:
            prefix_sum = prefix_sum + attention_output

        hidden_states = apply_kimi_attention_residual(
            prefix_sum.reshape(-1, hidden_dim),
            block_residual,
            self.mlp_res_proj,
            self.mlp_res_norm,
        )
        hidden_states = hidden_states.view(
            batch_size,
            sequence_length,
            hidden_dim,
        )

        hidden_states = self.post_attention_norm(hidden_states)
        mlp_output = self.mlp_branch(hidden_states)

        prefix_sum = prefix_sum + mlp_output
        return prefix_sum, block_residual


class K3AttentionResidualStack(nn.Module):
    def __init__(
        self,
        hidden_dim=8,
        depth=93,
        block_size=12,
    ):
        super().__init__()

        self.block_size = block_size

        self.layers = nn.ModuleList(
            [
                K3ResidualSite(
                    layer_index=layer_index,
                    hidden_dim=hidden_dim,
                    block_size=block_size,
                )
                for layer_index in range(depth)
            ]
        )

        self.output_res_norm = KimiRMSNorm(hidden_dim)
        self.output_res_proj = nn.Linear(
            hidden_dim,
            1,
            bias=False,
        )
        self.final_norm = KimiRMSNorm(hidden_dim)

    def forward(self, hidden_states):
        batch_size, sequence_length, hidden_dim = hidden_states.shape

        block_residual = hidden_states.new_zeros(
            batch_size * sequence_length,
            0,
            hidden_dim,
        )

        for layer in self.layers:
            hidden_states, block_residual = layer(
                hidden_states,
                block_residual,
            )

        hidden_states = apply_kimi_attention_residual(
            hidden_states.reshape(-1, hidden_dim),
            block_residual,
            self.output_res_proj,
            self.output_res_norm,
        )
        hidden_states = hidden_states.view(
            batch_size,
            sequence_length,
            hidden_dim,
        )

        return self.final_norm(hidden_states), block_residual


k3_residual_model = K3AttentionResidualStack().to(device)

assert len(k3_residual_model.layers) == 93
assert k3_residual_model.block_size == 12

k3_input = torch.randn(1, 2, 8, device=device)
k3_output, k3_bank = k3_residual_model(k3_input)

expected_bank_entries = (
    (93 - 1) // 12
    + 1
)
assert k3_bank.size(1) == expected_bank_entries

print("K3 residual sites:", len(k3_residual_model.layers))
print("K3 block size:", k3_residual_model.block_size)
print("K3 bank entries:", k3_bank.size(1))
print("K3 output:", k3_output.shape)

## Architecture audit

The residual algorithms now fail loudly if their released topology is shortened:

- DeepSeek-V4-Flash residual depth: 43.
- mHC expansion: 4 streams.
- Sinkhorn iterations: 20.
- Kimi K3 residual sites: 93.
- Kimi K3 residual block size: 12.
- Final K3 output Attention Residual mix is retained.

References: DeepSeek-V4 Transformers implementation and Moonshot Kimi-K3 released `modeling_kimi_linear.py`.